# Step 1.4: What do the labels mean? (SpeechJudge, MANGO, SOMOS)

**Run on the server** (kernel = `hinglishtts` venv, same as the first notebook). **Everything here is read-only.** Nothing is written, extracted or downloaded: SpeechJudge is read as text columns only (audio columns are never loaded), SOMOS files are read from inside the zip without extracting.

Why: loaders (Step 1.5) must not guess how ties, strength, scales and rater IDs are encoded. A wrong reading here would poison training, so we look at the real files first.

| Part | Dataset | Questions answered |
|---|---|---|
| A | SpeechJudge-Data | Splits and sizes; how ties and strength are encoded; how many raters per pair; language settings and the code-switched ("mixed") slice; does the test split contain only full-agreement pairs? |
| B | MANGO | Columns per CSV; score ranges per system (MUSHRA 0-100); reference and anchor behaviour; raters per stimulus |
| C | SOMOS | Zip layout; which file holds the scores; columns; whether rater IDs exist |
| D | Summary | One summary block for the ledger, also saved as `summary.json` |

The SpeechArenaBench label check (A / B / both good / both bad / ties) comes after the first notebook has put its Hindi data on disk.

In [ ]:
import collections
import json
import sys
import zipfile
from pathlib import Path

import pandas as pd
import pyarrow
import pyarrow.parquet

ROOT = Path.home() / "projects" / "hinglish-cs-tts"
DS = ROOT / "datasets"

# everything shown with log() is also saved to output.log, so no result lives only in kernel memory
OUTDIR = ROOT / "analysis" / "step1_4"
OUTDIR.mkdir(parents=True, exist_ok=True)


def log(*parts):
    """Print a line and append it to output.log."""
    line = " ".join([str(part) for part in parts])
    print(line)
    with open(OUTDIR / "output.log", "a") as log_file:
        log_file.write(line + "\n")


def save_json(name, data):
    """Save data as JSON under OUTDIR, atomically (temp file, then rename)."""
    tmp = OUTDIR / (name + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False, default=str))
    tmp.replace(OUTDIR / name)


log("python", sys.version.split()[0], "| pandas", pd.__version__, "| pyarrow", pyarrow.__version__)
for name in ["speechjudge", "mango", "somos"]:
    log(name, "exists:", (DS / name).exists())
log("outputs and results are saved under", OUTDIR)

## Part A: SpeechJudge-Data

In [ ]:
# A1. Parquet files per split (names look like <split>-00000-of-000NN.parquet) and the real schema
SJ = DS / "speechjudge"
shards = sorted(SJ.rglob("*.parquet"))
by_split = {}
for path in shards:
    split = path.name.split("-")[0]
    if split not in by_split:
        by_split[split] = []
    by_split[split].append(path)
for split, paths in by_split.items():
    size = 0
    for path in paths:
        size += path.stat().st_size
    log(f"{split:8} {len(paths):3} files {size / 1e9:7.1f} GB")

first = pyarrow.parquet.ParquetFile(by_split["test"][0])
log("\nschema:")
for field in first.schema_arrow:
    log(f"  {field.name:28} {field.type}")

In [ ]:
# A2. Load ONLY the text/label columns (audio columns are never read). Cached on disk as parquet so a restart does not redo it.
WANT = ["index", "subset", "language_setting", "chosen", "target_text", "rater",
        "audioA_text_accuracy", "audioB_text_accuracy", "naturalness_annotation", "naturalness_label"]
cols = [c for c in WANT if c in first.schema_arrow.names]
log("columns used:", cols, "| missing:", [c for c in WANT if c not in cols])

dfs = {}
for split, paths in by_split.items():
    cache = OUTDIR / f"sj_{split}.parquet"
    if cache.exists():
        dfs[split] = pd.read_parquet(cache)
        log(split, "rows (loaded from the disk cache):", len(dfs[split]))
        continue
    tables = []
    for path in paths:
        tables.append(pyarrow.parquet.read_table(path, columns=cols))
    df = pyarrow.concat_tables(tables).to_pandas()
    tmp = cache.with_suffix(".tmp")
    df.to_parquet(tmp)
    tmp.replace(cache)   # atomic: a half-written cache never carries the real name
    dfs[split] = df
    log(split, "rows:", len(df), "(saved to", cache.name + ")")
total_pairs = 0
for df in dfs.values():
    total_pairs += len(df)
log("total pairs:", total_pairs, "(dataset card says about 99K)")

In [ ]:
# A3. Labels, ties and strength, per split
def as_list(x):
    """A list cell as a Python list ([] when missing)."""
    if x is None:
        return []
    return list(x)


def list_length(x):
    """Number of items in a list cell."""
    return len(as_list(x))


def count_annotations(df):
    """How often each raw vote text (A+1, B+2, Tie ...) appears."""
    counts = collections.Counter()
    for annotations in df["naturalness_annotation"]:
        for annotation in as_list(annotations):
            counts[annotation] += 1
    return counts


def unique_raters(dfs):
    """All rater ids over all splits."""
    raters = set()
    for df in dfs.values():
        for rater_list in df["rater"]:
            for rater in as_list(rater_list):
                raters.add(rater)
    return raters


for split, df in dfs.items():
    log(f"\n=== {split} ({len(df)} pairs) ===")
    log("naturalness_label :", df["naturalness_label"].value_counts().to_dict())
    annotations = count_annotations(df)
    log("annotation values  :", dict(annotations))
    raters_per_pair = df["rater"].map(list_length)
    log("raters per pair    :", raters_per_pair.value_counts().sort_index().to_dict())
    log("language_setting   :", df["language_setting"].value_counts().to_dict())
    if "subset" in df:
        log("subset             :", df["subset"].value_counts().to_dict())
    if "chosen" in df:
        log("chosen (agree>40%) :", df["chosen"].value_counts().to_dict())
    strong = 0
    weak = 0
    for annotation, count in annotations.items():
        if annotation.endswith("+2"):
            strong += count
        if annotation.endswith("+1"):
            weak += count
    log(f"strength           : +1 = {weak}, +2 = {strong}")
log("\nunique rater ids across splits:", len(unique_raters(dfs)))

In [ ]:
# A4. Does 'test' really contain only full-agreement pairs? And how is the label derived from raw votes?
def count_votes(annotations):
    """(votes for A, votes for B, tie votes) in one annotation list."""
    a = 0
    b = 0
    tie = 0
    for vote in as_list(annotations):
        if vote.startswith("A"):
            a += 1
        if vote.startswith("B"):
            b += 1
        if vote.startswith("Tie"):
            tie += 1
    return (a, b, tie)


def is_full_agreement(votes):
    """True if all raters chose the same side with no ties, or all raters chose a tie."""
    a, b, tie = votes
    one_side = (a == 0) != (b == 0) and tie == 0
    all_tie = a == 0 and b == 0 and tie > 0
    return one_side or all_tie


def majority_label(votes):
    """A or B if that side has more votes than the other side and than the ties, otherwise Tie."""
    a, b, tie = votes
    if a > b and a > tie:
        return "A"
    if b > a and b > tie:
        return "B"
    return "Tie"


for split, df in dfs.items():
    votes = df["naturalness_annotation"].map(count_votes)
    full = votes.map(is_full_agreement)
    log(f"{split:6} full agreement (all raters same side, no ties mixed in): {int(full.sum())} of {len(df)} ({100 * full.mean():.0f}%)")

# majority-vote check on the 'other' split (or the first split if there is none)
if "other" in dfs:
    df = dfs["other"]
    checked_split = "other"
else:
    df = list(dfs.values())[0]
    checked_split = "first"
votes = df["naturalness_annotation"].map(count_votes)
agree = (votes.map(majority_label) == df["naturalness_label"]).mean()
log(f"\nlabel equals simple majority of votes in {100 * agree:.0f}% of rows of split '{checked_split}'")
log("a few rows:")
log(df[["naturalness_annotation", "naturalness_label"]].head(5).to_string())

In [ ]:
# A5. The code-switched ("mixed") slice: how many pairs per split, and its label balance
for split, df in dfs.items():
    mixed = df["language_setting"].str.contains("mixed")
    labels = df[mixed]["naturalness_label"].value_counts().to_dict()
    log(f"{split:6} mixed pairs: {int(mixed.sum()):6d} of {len(df)} | labels {labels}")

## Part B: MANGO

In [ ]:
MG = DS / "mango"
mango = {}
for path in sorted((MG / "csvs").glob("*.csv")):
    df = pd.read_csv(path)
    mango[path.name] = df
    score_cols = [c for c in df.columns if c.endswith("_Score")]
    audio_cols = [c for c in df.columns if c.endswith("_Audio")]
    log(f"\n=== {path.name}: {len(df)} rows, raters {df['Rater_ID'].nunique()} ===")
    log("columns:", list(df.columns))
    log(df[score_cols].describe().loc[["min", "mean", "max"]].round(1).to_string())
    if "REF_Audio" in df:
        ref = "REF_Audio"
    else:
        ref = audio_cols[0]
    log(f"distinct stimuli ({ref}): {df[ref].nunique()} | ratings per stimulus (mean): {len(df) / df[ref].nunique():.1f}")

In [ ]:
# B2. How do systems compare on average (this tells us how to turn MUSHRA pages into pairs), per CSV
for name, df in mango.items():
    means = {}
    for column in df.columns:
        if column.endswith("_Score"):
            means[column.replace("_Score", "")] = round(float(df[column].mean()), 1)
    log(name, means)
log("\nreminder: audio folders also hold an XTTS system; check above whether any CSV scores it")

## Part C: SOMOS (read from inside the zip, nothing extracted)

In [ ]:
z = zipfile.ZipFile(DS / "somos" / "somos.zip")
infos = z.infolist()
names = [info.filename for info in infos]
compressed = 0
for info in infos:
    compressed += info.compress_size
log("entries:", len(names), "| compressed total GB:", round(compressed / 1e9, 2))

log("\nfolders (first 3 levels):")
top = collections.Counter()
extensions = collections.Counter()
for name in names:
    if not name.endswith("/"):
        top["/".join(name.split("/")[:3])] += 1
        extensions[Path(name).suffix] += 1
for key, count in top.most_common(20):
    log(f"{count:7d}  {key}")
log("\nextensions:", extensions.most_common(8))

TEXT_ENDINGS = (".csv", ".tsv", ".txt", ".json", ".md")
texts = []
for info in infos:
    if info.filename.lower().endswith(TEXT_ENDINGS) and info.file_size < 50_000_000:
        texts.append(info)
log("\ntext-like files:")
for info in texts:
    log(f"{info.file_size / 1e6:9.2f} MB  {info.filename}")

In [ ]:
# C2. Header and first rows of every small text-like file (so we find the score table and its columns)
for info in texts[:12]:
    with z.open(info.filename) as fh:
        head = fh.read(1500).decode("utf-8", errors="replace")
    log(f"\n######## {info.filename}")
    log("\n".join(head.splitlines()[:6])[:900])

In [ ]:
# C3. If a score table was found above, set its name here and re-run this cell
SCORES_FILE = None      # e.g. "somos/training_files/split1/full/train_mos_list.txt"  (fill in from the listing above)
if SCORES_FILE:
    with z.open(SCORES_FILE) as fh:
        sdf = pd.read_csv(fh)
    log(sdf.shape)
    log(sdf.head())
    log(sdf.describe(include="all").T.head(15))
else:
    log("SCORES_FILE is None: pick the score table from the listing above, set it, and re-run.")

## Part D: summary for the ledger

In [ ]:
split_sizes = {}
labels = {}
mixed_pairs = {}
for split, df in dfs.items():
    split_sizes[split] = len(df)
    labels[split] = df["naturalness_label"].value_counts().to_dict()
    mixed_pairs[split] = int(df["language_setting"].str.contains("mixed").sum())
mango_rows_raters = {}
for name, df in mango.items():
    mango_rows_raters[name] = [len(df), int(df["Rater_ID"].nunique())]
text_names = [info.filename for info in texts]

log("=== SUMMARY ===")
log("SpeechJudge splits:", split_sizes)
for split, df in dfs.items():
    log(split, "| labels", labels[split], "| mixed pairs", mixed_pairs[split], "| annotation values", dict(count_annotations(df)))
log("SpeechJudge unique raters:", len(unique_raters(dfs)))
log("MANGO csvs:", mango_rows_raters)
log("SOMOS zip entries:", len(names), "| text files:", text_names[:12])
log("=== END ===")

# also save the summary as a file
summary = {
    "speechjudge_splits": split_sizes,
    "speechjudge_labels": labels,
    "speechjudge_mixed_pairs": mixed_pairs,
    "speechjudge_unique_raters": len(unique_raters(dfs)),
    "mango_csvs_rows_raters": mango_rows_raters,
    "somos_zip_entries": len(names),
    "somos_text_files": text_names,
}
save_json("summary.json", summary)
log("summary saved to", OUTDIR / "summary.json")